# Final model analysis

In [ ]:
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, f1_score
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer


from src.classifiers import load_best_trial
from src.config import (
    CLEAN_DATA_PATH,
    DEVICE,
    FASTTEXT_MODEL_PATH,
    RESULTS_PATH,
    SEED,
    TRANSFORMER_MAX_LENGTH,
)
from src.embeddings import embed_documents, load_fasttext
from src.evaluate import evaluate
from src.transformer import TextClassificationDataset


In [ ]:
TRANSFORMER_DIR = RESULTS_PATH / "transformer" / "ufal_robeczech-base__tuned"
FIGURES_PATH = RESULTS_PATH / "figures"
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

Load the data

In [ ]:
df = pd.read_parquet(CLEAN_DATA_PATH)
test = df[df["split"] == "test"].reset_index(drop=True)
y_test_names = test["category"].values
len(test)

Helper function for measuring the size of models

In [ ]:
def size_mb(path: Path) -> float:
    """Return the size of a file, or the total size of a directory, in MB."""
    path = Path(path)
    if path.is_file():
        return path.stat().st_size / 1024**2
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1024**2

Return function for validation macro-F1

In [ ]:
def best_val_macro_f1(trials_path: Path) -> float:
    """Return the validation macro-F1 of the winning classifier in a trials CSV."""
    _, score, _ = load_best_trial(trials_path)
    return score

### Perform the analyses for best traditional model

In [ ]:
tfidf_model_path = RESULTS_PATH / "traditional_model.joblib"
tfidf_bundle = joblib.load(tfidf_model_path)
vectorizer = tfidf_bundle["vectorizer"]
tfidf_classifier = tfidf_bundle["classifier"]
tfidf_encoder = tfidf_bundle["encoder"]
vectorizer

In [ ]:
start = time.time()
X_test = vectorizer.transform(test["text"])
y_pred = tfidf_classifier.predict(X_test)
elapsed_tfidf = time.time() - start

pred_tfidf = tfidf_encoder.inverse_transform(y_pred)

In [ ]:
pd.DataFrame({"text": test["text"], "true": y_test_names, "pred": pred_tfidf}).to_csv(
    RESULTS_PATH / "traditional_test_predictions.csv", index=False
)

In [ ]:
tfidf_row = {
    "method": "tfidf",
    "val_macro_f1": best_val_macro_f1(RESULTS_PATH / "traditional_optuna_trials.csv"),
    **evaluate(y_test_names, pred_tfidf),
    "inference_seconds": elapsed_tfidf,
    "inference_ms_per_sample": elapsed_tfidf / len(test) * 1000,
    "model_size_mb": size_mb(tfidf_model_path),
}
tfidf_row

### Perform the analyses for best embedded model

In [ ]:
ft_model_path = RESULTS_PATH / "embeddings_model.joblib"
ft_bundle = joblib.load(ft_model_path)
scaler = ft_bundle["scaler"]
ft_classifier = ft_bundle["classifier"]
ft_encoder = ft_bundle["encoder"]
fasttext_model = load_fasttext(FASTTEXT_MODEL_PATH)

In [ ]:
start = time.time()
X_test = embed_documents(test["text"], fasttext_model)
X_test = scaler.transform(X_test)
y_pred = ft_classifier.predict(X_test)
elapsed_ft = time.time() - start

pred_fasttext = ft_encoder.inverse_transform(y_pred)

In [ ]:
pd.DataFrame({"text": test["text"], "true": y_test_names, "pred": pred_fasttext}).to_csv(
    RESULTS_PATH / "embeddings_test_predictions.csv", index=False
)

In [ ]:
ft_row = {
    "method": "fasttext",
    "val_macro_f1": best_val_macro_f1(RESULTS_PATH / "embeddings_optuna_trials.csv"),
    **evaluate(y_test_names, pred_fasttext),
    "inference_seconds": elapsed_ft,
    "inference_ms_per_sample": elapsed_ft / len(test) * 1000,
    # The pretrained binary has to ship alongside the classifier, so it counts.
    "model_size_mb": size_mb(ft_model_path) + size_mb(FASTTEXT_MODEL_PATH),
}
ft_row

### Perform the analyses for best transformer model

In [ ]:
model_dir = TRANSFORMER_DIR / "best"

encoder_classes = np.load(TRANSFORMER_DIR / "label_classes.npy", allow_pickle=True)
tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
model.to(DEVICE).eval()

In [ ]:
# Numeric labels for the dataset (uses the same order as label_classes.npy).
name_to_id = {name: i for i, name in enumerate(encoder_classes)}
y_test_ids = np.array([name_to_id[name] for name in y_test_names])
dataset = TextClassificationDataset(test["text"], y_test_ids, tokenizer, TRANSFORMER_MAX_LENGTH)
loader = DataLoader(dataset, batch_size=64, shuffle=False)

In [ ]:
predictions = []
start = time.time()
with torch.no_grad():
    for batch in loader:
        logits = model(
            input_ids=batch["input_ids"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE),
        ).logits
        predictions.extend(logits.argmax(dim=-1).cpu().tolist())
elapsed_tr = time.time() - start

pred_transformer = encoder_classes[predictions]

In [ ]:
pd.DataFrame({"text": test["text"], "true": y_test_names, "pred": pred_transformer}).to_csv(
    TRANSFORMER_DIR / "test_predictions.csv", index=False
)

In [ ]:
val_metrics = pd.read_csv(TRANSFORMER_DIR / "val_metrics.csv")

tr_row = {
    "method": "transformer",
    "val_macro_f1": val_metrics["macro_f1"].iloc[0],
    **evaluate(y_test_names, pred_transformer),
    "inference_seconds": elapsed_tr,
    "inference_ms_per_sample": elapsed_tr / len(test) * 1000,
    "model_size_mb": size_mb(model_dir),
}
tr_row

### Comparison table

In [ ]:
comparison = pd.DataFrame([tfidf_row, ft_row, tr_row]).sort_values(
    "macro_f1", ascending=False
)
comparison.to_csv(RESULTS_PATH / "comparison.csv", index=False)
comparison